In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

intents = {
    "greeting": {
        "patterns": ["hi", "hello", "hey", "hola", "greetings"],
        "responses": ["Hello! Tell me about your skills and experience, and I'll recommend a job role."]
    },
    "goodbye": {
        "patterns": ["bye", "see you", "goodbye"],
        "responses": ["Goodbye! Good luck with your job search!"]
    },
    "thanks": {
        "patterns": ["thanks", "thank you", "thx"],
        "responses": ["You’re welcome!", "Glad I could help!"]
    },
    "help": {
        "patterns": ["can you help me", "i need assistance", "help me"],
        "responses": ["Sure! Tell me your skills and experience, and I'll suggest a suitable job role."]
    }
}
def detect_intent(user_text):
    user_text = user_text.lower()
    for intent, data in intents.items():
        for pattern in data["patterns"]:
            if pattern in user_text:
                return intent
    return "job_recommendation"  # default intent if no match


# 1. Load dataset
df = pd.read_csv("candidate_job_role_dataset.csv")  # Replace with actual path

# 2. Preprocess features

# 2a. Extract all unique skills
all_skills = sorted(list({skill.strip().lower() 
                          for row in df['skills'] 
                          for skill in row.split(',')}))

def skills_to_vector(skills_text):
    skills_text = skills_text.lower()
    return np.array([1 if s in skills_text else 0 for s in all_skills])

X_skills = np.array([skills_to_vector(s) for s in df['skills']])

# 2b. Qualification one-hot encoding
qual_le = LabelEncoder()
qual_labels = qual_le.fit_transform(df['qualification'])
qual_ohe = OneHotEncoder(sparse_output=False)
X_qual = qual_ohe.fit_transform(qual_labels.reshape(-1,1))

# 2c. Experience level one-hot encoding (Junior/Mid/Senior)
exp_le = LabelEncoder()
exp_labels = exp_le.fit_transform(df['experience_level'])
exp_ohe = OneHotEncoder(sparse_output=False)
X_exp = exp_ohe.fit_transform(exp_labels.reshape(-1,1))

# Combine features
X = np.hstack([X_skills, X_qual, X_exp])

# 2d. Encode target job role
job_le = LabelEncoder()
y_labels = job_le.fit_transform(df['job_role'])
num_classes = len(job_le.classes_)
y = np.zeros((len(y_labels), num_classes))
for i, lab in enumerate(y_labels):
    y[i, lab] = 1

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Build ANN (improved with BatchNorm)
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 5. Train with early stopping
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train, y_train, validation_split=0.2, epochs=200, batch_size=32, callbacks=[early], verbose=1)

# Evaluate model
loss, acc = model.evaluate(X_test, y_test, verbose=1)
print(f"Test accuracy: {acc*100:.2f}%")

# 6. Chatbot function
def predict_job_from_text(user_text, exp_input):
    user_text_lower = user_text.lower()
    
    # Skills
    skill_vec = np.array([1 if s in user_text_lower else 0 for s in all_skills]).reshape(1,-1)
    
    # Qualification
    qual_found = None
    for q in qual_le.classes_:
        if q.lower() in user_text_lower:
            qual_found = q
            break
    if qual_found is None:
        qual_found = qual_le.classes_[0]  # default
    qual_vec = qual_ohe.transform([[qual_le.transform([qual_found])[0]]])
    
    # Experience level
    try:
        exp_label = exp_le.transform([exp_input])[0]
    except:
        exp_label = exp_le.transform([exp_le.classes_[0]])[0]  # default to first
    exp_vec = exp_ohe.transform([[exp_label]])
    
    # Combine
    x_input = np.hstack([skill_vec, qual_vec, exp_vec])
    
    # Predict
    pred = model.predict(x_input, verbose=0)[0]
    idx = np.argmax(pred)
    job_role = job_le.inverse_transform([idx])[0]
    return job_role, pred

# 7. Chat loop
print("Type 'exit' to quit.")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    
    intent = detect_intent(user_input)
    
    if intent != "job_recommendation":
        # Pick a random response from the intent
        responses = intents[intent]["responses"]
        print("Chatbot:", np.random.choice(responses))
        continue
    
    # Job recommendation flow
    exp = input("Enter your experience level (Junior/Mid/Senior): ")
    job, probs = predict_job_from_text(user_input, exp)
    print(f"Chatbot: Recommended Job Role → {job}")
    
    # Optional: show top 3 predictions
    top_idx = np.argsort(probs)[-3:][::-1]
    top_jobs = {job_le.inverse_transform([i])[0]: float(probs[i]) for i in top_idx}
    print("Top suggestions:", top_jobs)

